# 🎨 Fizgig LoRA Trainer — free Colab edition (Tesla T4)

Trains LoRAs with **[Fizgig](https://github.com/shootthesound/Fizgig)** (Klein, Flux 3 Klein, Krea 2, Qwen Image 2.1, Z-Image, SDXL, Anima …) completely free on Colab.

**How to use**
1. `Runtime ▸ Change runtime type ▸ Hardware accelerator: T4 GPU`
2. `Runtime ▸ Run all` (or run cells ①→⑦ in order). Cell ① has all options; every option is optional.
3. Cell ⑦ prints a **public link + password**. Open it, then in Fizgig: *Preferences ▸ Download models for me*, *Start ▸ Upload a training folder*, *Training ▸ Start*.
4. **Leave cell ⑦ running** (it is the monitor + keep-alive). Stop it with ■ to run utilities; the server keeps running in the background and re-running ⑦ re-attaches.

**Every new session automatically installs the newest Fizgig (git) and its newest model list.**
Pin a version by setting `FIZGIG_REF` (e.g. `v9.0.8`) in cell ①.

> ⚠️ Free Colab = 12 GB RAM. Pick models marked *Smooth/Good* in the table on the web page, keep **batch size 1**, leave precision on **Auto**, and use few sample previews.

In [ ]:
#@title ① Settings — edit if you like, then run { display-mode: "form" }
#@markdown Everything is optional. Defaults work on a free **T4** runtime.
#@markdown ---
#@markdown ### 📁 Storage (Google Drive is optional)
USE_DRIVE = False #@param {type:"boolean"}
DRIVE_FOLDER = "Fizgig" #@param {type:"string"}
#@markdown Keep models on Drive too? (needs 30+ GB free Drive space — usually leave OFF)
STORE_MODELS_ON_DRIVE = False #@param {type:"boolean"}
#@markdown ---
#@markdown ### 🔄 Version (every session pulls the newest by default)
FIZGIG_REF = "master" #@param {type:"string"}
FIZGIG_REPO = "https://github.com/shootthesound/Fizgig.git" #@param {type:"string"}
FORCE_REINSTALL = False #@param {type:"boolean"}
#@markdown ---
#@markdown ### 🔑 Tokens (optional — Colab Secrets `HF_TOKEN` / `NGROK_AUTHTOKEN` are also read)
HF_TOKEN = "" #@param {type:"string"}
NGROK_TOKEN = "" #@param {type:"string"}
#@markdown Login password for the web UI (≥ 8 chars). Leave empty to auto-generate.
FIZGIG_PASSWORD = "" #@param {type:"string"}
#@markdown ---
#@markdown ### 🌐 Public tunnel
#@markdown `auto` = Cloudflare → ngrok (only if token) → localhost.run → Pinggy
TUNNEL = "auto" #@param ["auto","cloudflared","ngrok","localhost.run","pinggy"]
PORT = 8081 #@param {type:"integer"}
#@markdown ---
#@markdown ### 🛡️ Keep-alive & stability
KEEPALIVE_JS = True #@param {type:"boolean"}
KEEPALIVE_AUDIO = True #@param {type:"boolean"}
AUTO_RESTART_SERVER = True #@param {type:"boolean"}
#@markdown ---
#@markdown ### ⚙️ Extras
#@markdown Swap file in GB (helps the 12 GB RAM limit; skipped automatically if Colab forbids it). 0 = off.
SWAP_GB = 8 #@param {type:"slider", min:0, max:16, step:2}
#@markdown Download models before launch, e.g. `krea2` or `krea2,tools` (families: krea2, klein, flux3, minimax, qwen_image21, tools). Empty = do it later in the UI.
PREFETCH_MODELS = "" #@param {type:"string"}

import os, re, textwrap

class FizgigError(Exception):
    """Readable error: shows the problem + how to fix it, without a scary traceback."""
    def __init__(self, problem, fix="", detail=""):
        super().__init__(problem)
        self.problem, self.fix, self.detail = problem, fix, detail
    def _render_traceback_(self):
        out = ["\x1b[1;31m❌ " + self.problem + "\x1b[0m"]
        if self.fix:
            out.append("\x1b[1;33m👉 How to fix:\x1b[0m")
            out += ["   " + l for l in textwrap.dedent(self.fix).strip().splitlines()]
        if self.detail:
            out += ["", "\x1b[90m--- details (last lines) ---\x1b[0m", self.detail[-2000:]]
        return out

def _secret(*names):
    try:
        from google.colab import userdata
    except Exception:
        return ""
    for n in names:
        try:
            v = userdata.get(n)
            if v:
                return str(v).strip()
        except Exception:
            pass
    return ""

_hf = str(HF_TOKEN).strip() or _secret("HF_TOKEN", "HUGGINGFACE_TOKEN") or os.environ.get("HF_TOKEN", "")
_ng = str(NGROK_TOKEN).strip() or _secret("NGROK_AUTHTOKEN", "NGROK_TOKEN")

if not re.fullmatch(r"[A-Za-z0-9._/\-]+", str(FIZGIG_REF).strip() or "!"):
    raise FizgigError("FIZGIG_REF looks invalid: %r" % FIZGIG_REF, "Use a branch or tag name such as  master  or  v9.0.8  (no spaces).")
if not (1024 <= int(PORT) <= 65535):
    raise FizgigError("PORT must be between 1024 and 65535", "Set PORT = 8081 (default).")
if ".." in str(DRIVE_FOLDER):
    raise FizgigError("DRIVE_FOLDER must be a simple folder name", "Example: Fizgig")
if FIZGIG_PASSWORD.strip() and len(FIZGIG_PASSWORD.strip()) < 8:
    raise FizgigError("FIZGIG_PASSWORD must have at least 8 characters", "Make it longer, or leave it empty to auto-generate a strong one.")
if str(PREFETCH_MODELS).strip() and not re.fullmatch(r"[a-z0-9_,\s]+", str(PREFETCH_MODELS).strip().lower()):
    raise FizgigError("PREFETCH_MODELS has invalid characters", "Use comma-separated family names, e.g.  krea2,tools")

CFG = dict(
    USE_DRIVE=bool(USE_DRIVE), DRIVE_FOLDER=str(DRIVE_FOLDER).strip() or "Fizgig",
    STORE_MODELS_ON_DRIVE=bool(STORE_MODELS_ON_DRIVE),
    FIZGIG_REF=str(FIZGIG_REF).strip(), FIZGIG_REPO=str(FIZGIG_REPO).strip(),
    FORCE_REINSTALL=bool(FORCE_REINSTALL),
    HF_TOKEN=_hf, NGROK_TOKEN=_ng, FIZGIG_PASSWORD=FIZGIG_PASSWORD.strip(),
    TUNNEL=str(TUNNEL), PORT=int(PORT),
    KEEPALIVE_JS=bool(KEEPALIVE_JS), KEEPALIVE_AUDIO=bool(KEEPALIVE_AUDIO),
    AUTO_RESTART_SERVER=bool(AUTO_RESTART_SERVER),
    SWAP_GB=int(SWAP_GB), PREFETCH_MODELS=str(PREFETCH_MODELS).strip().lower(),
)
print("✅ Settings saved")
print("   Drive:", "ON (%s)" % CFG["DRIVE_FOLDER"] if CFG["USE_DRIVE"] else "OFF (local session storage)")
print("   Version:", CFG["FIZGIG_REF"], "| tunnel:", CFG["TUNNEL"], "| ngrok token:", "yes" if _ng else "no", "| HF token:", "yes" if _hf else "no")
print("   KEEPALIVE_JS=%s  KEEPALIVE_AUDIO=%s  AUTO_RESTART_SERVER=%s" % (CFG["KEEPALIVE_JS"], CFG["KEEPALIVE_AUDIO"], CFG["AUTO_RESTART_SERVER"]))

In [ ]:
#@title ② Preflight check (GPU · RAM · disk · internet)
if "CFG" not in globals():
    raise RuntimeError("▶ Run cell ① (Settings) first.")

import os, sys, re, json, time, shutil, subprocess, textwrap, socket, hashlib, secrets, platform
import urllib.request, urllib.error
from pathlib import Path
from urllib.parse import urlparse

ROOT = Path("/content")
APP = ROOT / "Fizgig"
VENV = ROOT / "fizgig_venv"
BIN = ROOT / "bin"
ROOT.mkdir(parents=True, exist_ok=True)
BIN.mkdir(parents=True, exist_ok=True)

def ensure(*names):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise FizgigError("A previous cell has not been run yet",
                          "Run the numbered cells in order, starting at ① (Runtime ▸ Run all).\nMissing: " + ", ".join(missing))

def run(cmd, env=None, cwd=None, quiet=False, problem=None, fix=None):
    """Run a command, stream its output. Raises FizgigError only when `problem` is given."""
    try:
        p = subprocess.Popen(cmd, shell=isinstance(cmd, str), cwd=cwd, env=env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, errors="replace", bufsize=1)
    except (FileNotFoundError, PermissionError) as e:
        if problem:
            raise FizgigError(problem, fix or "A required program is missing.", str(e))
        return 127, str(e)
    tail = []
    for line in p.stdout:
        line = line.rstrip()
        if not line:
            continue
        tail.append(line)
        if len(tail) > 80:
            tail.pop(0)
        if not quiet:
            print("   " + line[:220])
    rc = p.wait()
    out = "\n".join(tail)
    if rc != 0 and problem:
        raise FizgigError(problem, fix or "See the details below.", out)
    return rc, out

def http_status(url, timeout=6):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (FizgigColab)"})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status
    except urllib.error.HTTPError as e:
        return e.code
    except Exception:
        return None

def venv_env(extra=None):
    e = os.environ.copy()
    # Colab's system CUDA libs must not shadow the CUDA runtime bundled in the PyTorch wheels
    ld = [p for p in e.get("LD_LIBRARY_PATH", "").split(":") if p and "/cuda" not in p]
    e["LD_LIBRARY_PATH"] = ":".join(ld)
    e["VIRTUAL_ENV"] = str(VENV)
    e["PATH"] = str(VENV / "bin") + ":" + e.get("PATH", "")
    for k in ("PYTHONHOME", "PYTHONPATH"):
        e.pop(k, None)
    e.update(UV_CACHE_DIR=str(ROOT / ".uv_cache"), UV_NO_PROGRESS="1", NO_COLOR="1", PIP_PROGRESS_BAR="off",
             PIP_DISABLE_PIP_VERSION_CHECK="1", UV_HTTP_TIMEOUT="300", PYTHONUNBUFFERED="1")
    if extra:
        e.update(extra)
    return e

def _meminfo_gb():
    try:
        for l in open("/proc/meminfo"):
            if l.startswith("MemTotal"):
                return int(l.split()[1]) / 1024 / 1024
    except Exception:
        pass
    return 0.0

def preflight():
    print("🔎 Preflight")
    print("   Python %s on %s" % (sys.version.split()[0], platform.platform()))
    if "google.colab" not in sys.modules:
        print("   ⚠️ This does not look like Google Colab — it may still work on any Linux box with an NVIDIA GPU.")

    rc, out = run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"], quiet=True)
    if rc != 0 or "," not in out:
        raise FizgigError("No NVIDIA GPU found in this runtime",
            """
            1. Menu: Runtime ▸ Change runtime type ▸ Hardware accelerator ▸ T4 GPU ▸ Save
            2. Re-run from cell ① (switching runtime type restarts the VM).
            If T4 is greyed out, Colab has no free GPU at the moment: wait 15–60 min and retry.""", out)
    name, mem, drv = [x.strip() for x in out.splitlines()[0].split(",")][:3]
    vram = float(mem) / 1024
    print("   GPU: %s | %.1f GB VRAM | driver %s" % (name, vram, drv))
    if vram < 8:
        raise FizgigError("GPU has only %.1f GB VRAM (Fizgig needs 8 GB or more)" % vram,
                          "Use a T4 runtime (Runtime ▸ Change runtime type).")
    try:
        if int(drv.split(".")[0]) < 525:
            print("   ⚠️ NVIDIA driver %s is old; CUDA 12.8 wheels may fail. If PyTorch cannot see the GPU, restart the runtime." % drv)
    except ValueError:
        pass
    if "T4" in name:
        print("   ✅ Tesla T4 detected — NF4/INT8 paths will be used (no native bf16/fp8 on Turing).")

    ram = _meminfo_gb()
    print("   RAM: %.1f GB" % ram)
    if ram and ram < 10:
        print("   ⚠️ Less than 10 GB RAM: only small models (SDXL, Anima, Z-Image) are advisable.")

    free = shutil.disk_usage("/content").free / 2**30
    print("   Disk free: %.0f GB" % free)
    if free < 25:
        raise FizgigError("Only %.0f GB of disk space is free" % free,
            """
            Runtime ▸ Disconnect and delete runtime, then start again (a fresh VM has ~70 GB).
            Delete big files in /content (left panel ▸ Files) if you uploaded some.""")
    if free < 45:
        print("   ⚠️ Under 45 GB free: install + one model family needs ~45 GB. Use a fresh runtime if downloads fail.")

    for url, label in (("https://github.com", "GitHub"), ("https://download.pytorch.org/whl/cu128/", "PyTorch wheels"), ("https://huggingface.co", "Hugging Face")):
        s = http_status(url, 10)
        if s is None:
            raise FizgigError("Cannot reach %s from this runtime" % label,
                              "Colab's network may be having a hiccup. Wait a minute and re-run this cell; if it persists use Runtime ▸ Disconnect and delete runtime.")
    print("   ✅ Internet OK (GitHub, PyTorch, Hugging Face)")

preflight()
print("\n✅ Preflight passed")

In [ ]:
#@title ③ Storage — local, or Google Drive (optional)
if "ensure" not in globals():
    raise RuntimeError("▶ Run the cells above first (Runtime ▸ Run all).")

if CFG["USE_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        raise FizgigError("Google Drive could not be mounted",
            """
            • Click “Connect to Google Drive” in the popup and allow access, then re-run this cell.
            • Or skip Drive: untick USE_DRIVE in cell ① and re-run ① → ③ (everything still works,
              you just download your LoRAs from Fizgig's “Runs & outputs” tab or with the Export cell).""", str(e))
    if not Path("/content/drive/MyDrive").exists():
        raise FizgigError("Drive mounted but /content/drive/MyDrive was not found",
                          "Run  drive.mount('/content/drive', force_remount=True)  in a new cell, then re-run this cell.")
    WORKSPACE = Path("/content/drive/MyDrive") / CFG["DRIVE_FOLDER"]
else:
    WORKSPACE = Path("/content/fizgig_workspace")

try:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
except Exception as e:
    raise FizgigError("Cannot create the workspace folder %s" % WORKSPACE, "Check Drive permissions / free space, or set USE_DRIVE off.", str(e))

DATASETS = WORKSPACE / "datasets"
OUTPUTS = WORKSPACE / "output_loras"
PRESETS = WORKSPACE / "presets"
MODELS = (WORKSPACE / "models") if (CFG["USE_DRIVE"] and CFG["STORE_MODELS_ON_DRIVE"]) else Path("/content/fizgig_models")
HF_HOME = Path("/content/hf_cache")
for d in (DATASETS, OUTPUTS, MODELS, HF_HOME):
    d.mkdir(parents=True, exist_ok=True)

print("✅ Storage ready")
print("   Workspace :", WORKSPACE, "(persistent on Drive)" if CFG["USE_DRIVE"] else "(temporary — lost when the session ends!)")
print("   Datasets  :", DATASETS)
print("   LoRA out  :", OUTPUTS)
print("   Models    :", MODELS, "(Drive)" if MODELS.is_relative_to(WORKSPACE) and CFG["USE_DRIVE"] else "(local disk, fast)")
if not CFG["USE_DRIVE"]:
    print("   💡 Without Drive, download your LoRAs before the session ends (Fizgig ▸ Runs & outputs ▸ Download, or the 📦 Export cell).")
if CFG["USE_DRIVE"] and CFG["STORE_MODELS_ON_DRIVE"]:
    try:
        gb = shutil.disk_usage("/content/drive").free / 2**30
        print("   Drive free: %.0f GB" % gb)
        if gb < 40:
            print("   ⚠️ Less than 40 GB free on Drive — models will not fit. Untick STORE_MODELS_ON_DRIVE in cell ①.")
    except Exception:
        pass

In [ ]:
#@title ④ Install / update Fizgig (newest code + models list from GitHub every session)
if "ensure" not in globals():
    raise RuntimeError("▶ Run the cells above first (Runtime ▸ Run all).")
ensure("WORKSPACE")
t0 = time.time()
BASE_ENV = venv_env()

# ---- 1) system packages (compiler + headers for insightface / torch.compile, ssh for tunnels) ----
print("📦 [1/5] System packages …")
run("apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends "
    "build-essential python3-dev libgl1 libglib2.0-0 openssh-client git curl ca-certificates", quiet=True)

# ---- 2) clone / update the repo ----
print("⬇️  [2/5] Fetching Fizgig (%s @ %s) …" % (CFG["FIZGIG_REPO"].rsplit("/", 1)[-1], CFG["FIZGIG_REF"]))

def sync_repo():
    ref, url = CFG["FIZGIG_REF"], CFG["FIZGIG_REPO"]
    if (APP / ".git").exists():
        run(["git", "-C", str(APP), "remote", "set-url", "origin", url], quiet=True)
        rc1, _ = run(["git", "-C", str(APP), "fetch", "--depth", "1", "origin", ref], quiet=True)
        rc2 = 1
        if rc1 == 0:
            rc2, _ = run(["git", "-C", str(APP), "reset", "--hard", "FETCH_HEAD"], quiet=True)
        if rc1 == 0 and rc2 == 0:
            return "updated"
        print("   ⚠️ Update failed — doing a clean re-clone")
        shutil.rmtree(APP, ignore_errors=True)
    rc, tail = run(["git", "clone", "--depth", "1", "--branch", ref, url, str(APP)], quiet=True)
    if rc != 0:
        raise FizgigError("Could not download Fizgig from GitHub",
            """
            • Check FIZGIG_REF in cell ① — it must be an existing branch or tag (default: master).
              Tags look like v9.0.8 (see github.com/shootthesound/Fizgig/releases).
            • Check FIZGIG_REPO (default https://github.com/shootthesound/Fizgig.git).
            • If GitHub is down, wait a few minutes and re-run this cell.""", tail)
    return "cloned"

state = sync_repo()
_, head = run(["git", "-C", str(APP), "log", "-1", "--format=%h  %cd  %s", "--date=short"], quiet=True)
print("   ✅ Fizgig %s → %s" % (state, head.strip()[:150]))
try:
    if "shootthesound/Fizgig" in CFG["FIZGIG_REPO"]:
        rq = urllib.request.Request("https://api.github.com/repos/shootthesound/Fizgig/releases/latest", headers={"User-Agent": "fizgig-colab"})
        with urllib.request.urlopen(rq, timeout=10) as r:
            print("   📰 Latest release on GitHub:", json.load(r).get("tag_name"))
except Exception:
    pass
for need in ("fizgig.py", "requirements.txt", "uv_install_deps.py", "src"):
    if not (APP / need).exists():
        raise FizgigError("The downloaded repository is missing '%s'" % need,
                          "FIZGIG_REF probably points to a very old/odd branch. Set FIZGIG_REF = master in cell ① and re-run ① → ④.")

# ---- 3) virtual environment + dependencies ----
req = APP / "requirements.txt"
req_hash = hashlib.sha1(req.read_bytes()).hexdigest()[:12]
marker = VENV / ".fizgig_deps"
healthy = (VENV / "bin" / "python").exists() and marker.exists() and marker.read_text().strip() == req_hash and not CFG["FORCE_REINSTALL"]

if healthy:
    print("🐍 [3/5] Dependencies already installed for this requirements.txt — skipping")
else:
    print("🐍 [3/5] Installing Python environment (first time per session: ~5–8 min, ~10 GB) …")
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], env=BASE_ENV, quiet=True,
        problem="Could not install the 'uv' package manager", fix="Re-run this cell; if it repeats, check your internet connection (Runtime ▸ Disconnect and delete runtime).")
    if VENV.exists():
        shutil.rmtree(VENV, ignore_errors=True)
    py_spec = sys.executable if (3, 10) <= sys.version_info[:2] <= (3, 13) else "3.12"
    e = venv_env(); e.pop("VIRTUAL_ENV", None)
    run([sys.executable, "-m", "uv", "venv", "--seed", "--python", py_spec, str(VENV)], env=e, quiet=True,
        problem="Could not create the Python virtual environment",
        fix="Re-run this cell. If it still fails, restart the runtime (Runtime ▸ Restart session) and run ① → ④ again.")
    venv_py = str(VENV / "bin" / "python")
    run([sys.executable, "-m", "uv", "pip", "install", "--python", venv_py, "uv"], env=BASE_ENV, quiet=True,
        problem="Could not install uv into the environment", fix="Re-run this cell.")

    installer = [venv_py, str(APP / "uv_install_deps.py"), str(req), str(VENV)]
    ok = False
    for attempt in (1, 2):
        rc, tail = run(installer, env=BASE_ENV, quiet=False)
        if rc == 0:
            ok = True
            break
        print("   ⚠️ install attempt %d failed — retrying …" % attempt)
    if not ok:
        print("   ⚠️ Retrying without the optional face-detection package (insightface) …")
        trimmed = ROOT / "requirements_no_insightface.txt"
        trimmed.write_text("\n".join(l for l in req.read_text().splitlines() if not re.match(r"\s*insightface", l)) + "\n")
        rc, tail = run([venv_py, str(APP / "uv_install_deps.py"), str(trimmed), str(VENV)], env=BASE_ENV, quiet=False)
        if rc != 0:
            raise FizgigError("Installing Fizgig's Python packages failed",
                """
                1. Re-run this cell (network hiccups are the most common cause).
                2. Tick FORCE_REINSTALL in cell ① and re-run ① → ④ for a clean environment.
                3. Out of disk space? Check the “Disk free” line in cell ② (need ~45 GB).
                4. If the last lines below mention a specific package, copy them into a GitHub issue.""", tail)
        print("   ℹ️ Installed without insightface (face-crop tools in Image Prep will be unavailable).")
        run([sys.executable, "-m", "uv", "pip", "install", "--python", venv_py, "insightface"], env=BASE_ENV, quiet=True)
    marker.write_text(req_hash)

venv_py = str(VENV / "bin" / "python")

# ---- 4) verify ----
print("🧪 [4/5] Verifying PyTorch + GPU …")
chk = ("import torch, transformers, diffusers, accelerate, safetensors;"
       "print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.is_available(),"
       " torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')")
rc, out = run([venv_py, "-c", chk], env=BASE_ENV, quiet=True,
              problem="Installed packages cannot be imported",
              fix="Tick FORCE_REINSTALL in cell ① and re-run ① → ④.")
print("   " + out.strip().splitlines()[-1])
if "gpu True" not in out:
    raise FizgigError("PyTorch cannot see the GPU",
        """
        1. Runtime ▸ Change runtime type ▸ T4 GPU (make sure a GPU is selected).
        2. Runtime ▸ Restart session, then run ① → ④ again.""", out)
rc, out = run([venv_py, "-c", "import bitsandbytes; print('bitsandbytes', bitsandbytes.__version__)"], env=BASE_ENV, quiet=True)
print("   " + (out.strip().splitlines()[-1] if rc == 0 else "⚠️ bitsandbytes import warning (4-bit/8-bit modes may be limited)"))
rc, out = run([venv_py, "-c", "import sys; sys.path.insert(0,'src'); import fizgig; print('fizgig package OK')"], env=BASE_ENV, cwd=str(APP), quiet=True,
              problem="Fizgig's own code cannot be imported", fix="Set FIZGIG_REF = master in cell ① and re-run ① → ④ (or tick FORCE_REINSTALL).")

# ---- 5) tidy ----
rc, out = run("du -sh %s 2>/dev/null | cut -f1" % VENV, quiet=True)
print("🧹 [5/5] Environment size: %s | disk free: %.0f GB | took %.1f min" % (out.strip(), shutil.disk_usage("/content").free / 2**30, (time.time() - t0) / 60))
print("\n✅ Fizgig is installed and up to date.")

In [ ]:
#@title ⑤ Link workspace, swap file & optional model prefetch
if "ensure" not in globals():
    raise RuntimeError("▶ Run the cells above first (Runtime ▸ Run all).")
ensure("WORKSPACE", "APP", "VENV")
if not (VENV / "bin" / "python").exists():
    raise FizgigError("Cell ④ has not finished successfully", "Run cell ④ (Install / update) and wait until it prints “Fizgig is installed”.")

def server_env(extra=None):
    e = venv_env({
        "PYTHONPATH": str(APP / "src"),
        "HF_HOME": str(HF_HOME),
        "XDG_CACHE_HOME": str(ROOT / ".cache"),
        "INSIGHTFACE_HOME": str(ROOT / ".insightface"),
        "HF_XET_CHUNK_CACHE_SIZE_BYTES": str(1 << 30),   # keep the xet chunk cache small (disk is tight)
        "TOKENIZERS_PARALLELISM": "false",
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    })
    if CFG["HF_TOKEN"]:
        e["HF_TOKEN"] = CFG["HF_TOKEN"]
        e["HUGGING_FACE_HUB_TOKEN"] = CFG["HF_TOKEN"]
    if extra:
        e.update(extra)
    return e

def link(target: Path, dest: Path):
    """Make `dest` (inside the repo) a symlink to `target` (workspace/Drive), keeping existing files."""
    target.mkdir(parents=True, exist_ok=True)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.is_symlink():
        if Path(os.readlink(dest)) == target:
            return
        dest.unlink()
    elif dest.exists():
        if dest.is_dir():
            for item in dest.iterdir():
                if not (target / item.name).exists():
                    shutil.move(str(item), str(target / item.name))
            shutil.rmtree(dest, ignore_errors=True)
        else:
            dest.unlink()
    dest.symlink_to(target, target_is_directory=True)

(APP / "user_data").mkdir(exist_ok=True)
link(MODELS, APP / "user_data" / "models")            # model downloads land here
if CFG["USE_DRIVE"]:
    link(OUTPUTS, APP / "output_loras")                # trained LoRAs persist on Drive
    link(PRESETS, APP / "user_data" / "presets")       # your saved presets persist on Drive

prefs_file = APP / "user_data" / "prefs.json"
try:
    prefs = json.loads(prefs_file.read_text()) if prefs_file.exists() else {}
except Exception:
    prefs = {}
prefs["input_dataset_dir"] = str(DATASETS)             # "Upload a training folder" goes here
prefs_file.write_text(json.dumps(prefs, indent=2))
print("✅ Workspace linked  (uploads → %s)" % DATASETS)

# ---- swap (best effort; some Colab sandboxes forbid it — that is fine) ----
def setup_swap(gb):
    if gb <= 0:
        return
    try:
        if len(Path("/proc/swaps").read_text().strip().splitlines()) > 1:
            print("✅ Swap already active")
            return
    except Exception:
        pass
    free = shutil.disk_usage("/content").free / 2**30
    if free < gb + 35:
        print("ℹ️ Swap skipped: not enough free disk (%.0f GB) — keeping space for models." % free)
        return
    sf = "/content/swapfile"
    rc, out = run("fallocate -l %dG %s && chmod 600 %s && mkswap %s >/dev/null && swapon %s" % (gb, sf, sf, sf, sf), quiet=True)
    if rc != 0:
        run("rm -f " + sf, quiet=True)
        print("ℹ️ Swap is not permitted in this Colab sandbox — continuing without it (this is not an error).")
    else:
        print("✅ %d GB swap enabled (extra safety net for the 12 GB RAM limit)" % gb)

setup_swap(CFG["SWAP_GB"])

# ---- optional model prefetch ----
fams = [f for f in re.split(r"[,\s]+", CFG["PREFETCH_MODELS"]) if f]
for fam in fams:
    free = shutil.disk_usage("/content").free / 2**30
    if free < 12:
        print("⚠️ Skipping '%s': only %.0f GB free." % (fam, free))
        continue
    if fam in ("klein", "flux3") and not CFG["HF_TOKEN"]:
        print("⚠️ '%s' is a gated model — it needs a free Hugging Face token (accept the licence on the model page, then set HF_TOKEN in cell ① or Colab Secrets). Skipping." % fam)
        continue
    print("⬇️  Downloading model family '%s' (resumable; this can take a while) …" % fam)
    rc, out = run([str(VENV / "bin" / "python"), "-m", "fizgig.scripts.fetch_models", "--family", fam], env=server_env(), cwd=str(APP))
    if rc != 0:
        print("⚠️ Download of '%s' failed — not fatal. You can retry in Fizgig: Preferences ▸ Download models for me." % fam)

print("""
┌─ Tips for a free T4 (15 GB VRAM / 12 GB RAM) ─────────────────────────────┐
│ • Smooth: SDXL, Anima, Z-Image · Good: Flux 3 Klein, Krea 2, Qwen Image   │
│ • Batch size 1, precision/base = Auto (NF4/INT8), few sample previews.    │
│ • Avoid on free Colab: MiniMax H3, Vimoo, Fibo, full fine-tunes (RAM).     │
│ • Download only the model family you train (disk ≈ 70 GB total).          │
└───────────────────────────────────────────────────────────────────────────┘""")

In [ ]:
#@title ⑥ Keep-alive: browser JS + silent audio (KEEPALIVE_JS / KEEPALIVE_AUDIO)
if "ensure" not in globals():
    raise RuntimeError("▶ Run the cells above first (Runtime ▸ Run all).")
import io, wave, base64, struct, random
from IPython.display import display, HTML, Javascript

KA_JS = r"""
(function () {
  if (window.__fizgigKA) { return; }
  window.__fizgigKA = true;
  var n = 0;
  function tick() {
    n++;
    try {  // works whenever Colab lets the output frame reach the toolbar
      var d = (window.parent && window.parent.document) || document;
      var b = d.querySelector('colab-connect-button');
      var t = b && b.shadowRoot && (b.shadowRoot.querySelector('#connect') || b.shadowRoot.querySelector('colab-toolbar-button'));
      if (t) { t.click(); }
    } catch (e) {}
    try { document.dispatchEvent(new MouseEvent('mousemove', {bubbles: true, clientX: 5 + (n % 7), clientY: 5})); } catch (e) {}
    try { google.colab.kernel.invokeFunction('fizgig.keepalive', [n], {}); } catch (e) {}
    var el = document.getElementById('fz-ka');
    if (el) { el.textContent = '\uD83D\uDC93 keep-alive ping #' + n + ' at ' + new Date().toLocaleTimeString(); }
  }
  tick();
  setInterval(tick, 45000);
})();
"""

AUDIO_JS = r"""
(function () {
  var a = document.getElementById('fz-audio');
  if (!a) { return; }
  a.volume = 0.05;
  var p = a.play();
  if (p && p.catch) {
    p.catch(function () {
      var h = document.getElementById('fz-ka-hint');
      if (h) { h.textContent = ' \u25B6 browser blocked autoplay: press play on the silent player once.'; }
    });
  }
})();
"""

def _silent_wav_b64(seconds=2, rate=8000):
    buf = io.BytesIO()
    with wave.open(buf, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(rate)
        # ±1 LSB noise: inaudible, but not digital silence (some browsers treat pure silence as "not playing")
        w.writeframes(b"".join(struct.pack("<h", random.choice((-1, 0, 1))) for _ in range(rate * seconds)))
    return base64.b64encode(buf.getvalue()).decode()

def start_keepalive():
    global KA_DONE
    if not (CFG["KEEPALIVE_JS"] or CFG["KEEPALIVE_AUDIO"]):
        print("ℹ️ Keep-alive disabled in cell ①")
        KA_DONE = True
        return
    try:
        from google.colab import output
        output.register_callback("fizgig.keepalive", lambda n=None: None)
    except Exception:
        pass
    html = '<div style="font:13px system-ui;color:#555;padding:6px 10px;border:1px dashed #aaa;border-radius:8px">'
    html += '<span id="fz-ka">keep-alive starting…</span><span id="fz-ka-hint" style="color:#b45309"></span>'
    if CFG["KEEPALIVE_AUDIO"]:
        html += '<br>🔇 silent looping audio: <audio id="fz-audio" controls loop autoplay style="height:26px;vertical-align:middle" '
        html += 'src="data:audio/wav;base64,' + _silent_wav_b64() + '"></audio>'
    html += "</div>"
    display(HTML(html))
    js = (KA_JS if CFG["KEEPALIVE_JS"] else "") + (AUDIO_JS if CFG["KEEPALIVE_AUDIO"] else "")
    display(Javascript(js))
    KA_DONE = True
    print("✅ Keep-alive active  (JS: %s, audio: %s). Keep this browser tab open." % (CFG["KEEPALIVE_JS"], CFG["KEEPALIVE_AUDIO"]))
    print("   The strongest keep-alive is cell ⑦ itself: a running cell keeps the kernel busy, so Colab does not treat it as idle.")

KA_DONE = False
start_keepalive()

In [ ]:
#@title ⑦ 🚀 Launch Fizgig + public tunnel — KEEP THIS CELL RUNNING
if "ensure" not in globals():
    raise RuntimeError("▶ Run the cells above first (Runtime ▸ Run all).")
ensure("WORKSPACE", "server_env", "start_keepalive")
if not (VENV / "bin" / "python").exists() or not (APP / "fizgig.py").exists():
    raise FizgigError("Fizgig is not installed yet", "Run cell ④ (Install / update) first.")

import html as _html
from IPython.display import display, HTML

PORT = CFG["PORT"]
NGROK = CFG["NGROK_TOKEN"]
LOG_SERVER = ROOT / "fizgig_server.log"
if "STATE" not in globals():
    STATE = dict(server=None, tproc=None, tname=None, url=None, restarts=0, t0=time.time())

# ------------------------------------------------------------------ password
def get_password():
    pw = CFG["FIZGIG_PASSWORD"]
    pf = ROOT / ".fizgig_password"          # local only (never written to Drive)
    if not pw and pf.exists():
        pw = pf.read_text().strip()
    if not pw:
        pw = secrets.token_urlsafe(12)
    try:
        pf.write_text(pw); os.chmod(pf, 0o600)
    except Exception:
        pass
    return pw

PASSWORD = get_password()

# ------------------------------------------------------------------ server
def port_open():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1)
        return s.connect_ex(("127.0.0.1", PORT)) == 0

def server_up():
    p = STATE["server"]
    return (p.poll() is None) if p is not None else port_open()

def log_tail(n=40):
    try:
        return "\n".join(LOG_SERVER.read_text(errors="replace").splitlines()[-n:])
    except Exception:
        return ""

def start_server():
    p = STATE["server"]
    if p is not None and p.poll() is None:
        return
    if port_open():
        STATE["server"] = None
        print("ℹ️ A Fizgig server is already listening on port %d — re-using it." % PORT)
        return
    env = server_env({"FIZGIG_PASSWORD": PASSWORD})
    cmd = [str(VENV / "bin" / "python"), "fizgig.py", "--web", "--host", "127.0.0.1", "--port", str(PORT)]
    with open(LOG_SERVER, "ab") as lf:
        lf.write(("\n===== start %s =====\n" % time.strftime("%Y-%m-%d %H:%M:%S")).encode())
        STATE["server"] = subprocess.Popen(cmd, cwd=str(APP), env=env, stdout=lf, stderr=subprocess.STDOUT, start_new_session=True)

def wait_ready(timeout=240):
    t = time.time()
    while time.time() - t < timeout:
        p = STATE["server"]
        if p is not None and p.poll() is not None:
            raise FizgigError("Fizgig exited while starting (exit code %s)" % p.returncode,
                """
                1. Tick FORCE_REINSTALL in cell ① and re-run ① → ④ (broken/partial install).
                2. Read the log lines below. “CUDA out of memory” / “Killed” = RAM or VRAM limit → restart the runtime and train a lighter model.
                3. Port busy? Change PORT in cell ① (e.g. 8090).""", log_tail())
        if http_status("http://127.0.0.1:%d/" % PORT, 3) is not None:
            return True
        time.sleep(2)
    raise FizgigError("Fizgig did not answer within %d s" % timeout,
                      "Run the 🧰 Utilities cell with “server log” to see what it is doing, then try “restart server”.", log_tail())

def kill_server():
    p = STATE["server"]
    if p is not None and p.poll() is None:
        p.terminate()
        try:
            p.wait(20)
        except subprocess.TimeoutExpired:
            p.kill()
    subprocess.run(["pkill", "-f", "fizgig.py --web"], check=False)
    STATE["server"] = None

# ------------------------------------------------------------------ tunnels
def _spawn(cmd, name, stdin=False):
    lf = ROOT / ("tunnel_%s.log" % name)
    f = open(lf, "wb")
    p = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT,
                         stdin=subprocess.PIPE if stdin else subprocess.DEVNULL, start_new_session=True)
    return p, lf

def _kill(p):
    try:
        if p is not None and p.poll() is None:
            p.terminate()
            try:
                p.wait(5)
            except subprocess.TimeoutExpired:
                p.kill()
    except Exception:
        pass

def _wait_url(p, lf, pattern, bad=(), timeout=60):
    rx = re.compile(pattern)
    t = time.time()
    while time.time() - t < timeout:
        txt = lf.read_text(errors="replace") if lf.exists() else ""
        for m in rx.finditer(txt):
            u = m.group(m.lastindex or 0)
            if (urlparse(u).hostname or "") not in bad:
                return u
        if p.poll() is not None:
            break
        time.sleep(1)
    return None

def _fail(p, lf, msg):
    _kill(p)
    tail = ""
    try:
        tail = " | ".join(lf.read_text(errors="replace").strip().splitlines()[-3:])
    except Exception:
        pass
    raise RuntimeError(msg + (" — " + tail[:300] if tail else ""))

def _download(url, dest):
    rc, _ = run(["curl", "-fsSL", "--retry", "3", "--connect-timeout", "20", "-o", str(dest), url], quiet=True)
    return rc == 0

def tunnel_cloudflared():
    exe = BIN / "cloudflared"
    if not exe.exists():
        if not _download("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", exe):
            raise RuntimeError("could not download cloudflared")
        os.chmod(exe, 0o755)
    p, lf = _spawn([str(exe), "tunnel", "--no-autoupdate", "--protocol", "http2", "--url", "http://127.0.0.1:%d" % PORT], "cloudflared")
    u = _wait_url(p, lf, r"https://[a-z0-9-]+\.trycloudflare\.com", bad=("api.trycloudflare.com",), timeout=90)
    if not u:
        _fail(p, lf, "no Cloudflare URL received")
    return p, u

def tunnel_ngrok():
    if not NGROK:
        raise RuntimeError("no ngrok token set")
    exe = BIN / "ngrok"
    if not exe.exists():
        got = False
        for url in ("https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz",
                    "https://ngrok-agent.s3.amazonaws.com/ngrok-v3-stable-linux-amd64.tgz"):
            if _download(url, BIN / "ngrok.tgz"):
                run(["tar", "-xzf", str(BIN / "ngrok.tgz"), "-C", str(BIN)], quiet=True)
                got = exe.exists()
                if got:
                    break
        if not got:
            raise RuntimeError("could not download ngrok")
        os.chmod(exe, 0o755)
    p, lf = _spawn([str(exe), "http", "http://127.0.0.1:%d" % PORT, "--authtoken", NGROK, "--log=stdout"], "ngrok")
    u = _wait_url(p, lf, r'started tunnel.*?url=(https://[^\s"]+)', timeout=45)
    if not u:
        _fail(p, lf, "ngrok gave no URL (check the token at dashboard.ngrok.com/get-started/your-authtoken; free accounts allow 1 session)")
    return p, u

def _ssh_base():
    if not shutil.which("ssh"):
        run("apt-get install -y -qq openssh-client", quiet=True)
    if not shutil.which("ssh"):
        raise RuntimeError("ssh client not available")
    return ["ssh", "-o", "StrictHostKeyChecking=no", "-o", "UserKnownHostsFile=/dev/null", "-o", "ServerAliveInterval=30",
            "-o", "ServerAliveCountMax=3", "-o", "ExitOnForwardFailure=yes", "-o", "ConnectTimeout=20"]

def tunnel_localhost_run():
    p, lf = _spawn(_ssh_base() + ["-R", "80:127.0.0.1:%d" % PORT, "nokey@localhost.run"], "localhost_run", stdin=True)
    u = _wait_url(p, lf, r"https://[a-zA-Z0-9-]+\.(?:lhr\.life|localhost\.run)", bad=("admin.localhost.run", "localhost.run"), timeout=45)
    if not u:
        _fail(p, lf, "no localhost.run URL received")
    return p, u

def tunnel_pinggy():
    p, lf = _spawn(_ssh_base() + ["-p", "443", "-R0:127.0.0.1:%d" % PORT, "free.pinggy.io"], "pinggy", stdin=True)
    u = _wait_url(p, lf, r"https://[a-zA-Z0-9.-]+\.pinggy[a-zA-Z-]*\.(?:link|io|online|cloud)",
                  bad=("dashboard.pinggy.io", "pinggy.io", "free.pinggy.io", "a.pinggy.io", "www.pinggy.io"), timeout=45)
    if not u:
        _fail(p, lf, "no Pinggy URL received")
    return p, u

TUNNELS = {"cloudflared": tunnel_cloudflared, "ngrok": tunnel_ngrok,
           "localhost.run": tunnel_localhost_run, "pinggy": tunnel_pinggy}

def tunnel_chain():
    chain = ["cloudflared"] + (["ngrok"] if NGROK else []) + ["localhost.run", "pinggy"]
    pref = CFG["TUNNEL"]
    if pref != "auto":
        if pref == "ngrok" and not NGROK:
            print("ℹ️ TUNNEL=ngrok but no ngrok token was given — using the default chain instead.")
        else:
            chain = [pref] + [c for c in chain if c != pref]
    return chain

def verify_url(url, timeout=75):
    t = time.time()
    while time.time() - t < timeout:
        s = http_status(url, 8)
        if s is not None and s < 500:
            return True
        time.sleep(3)
    return False

def stop_tunnel():
    _kill(STATE["tproc"])
    STATE.update(tproc=None, url=None, tname=None)

def start_tunnel():
    stop_tunnel()
    errors = []
    for name in tunnel_chain():
        print("🌐 Opening %s tunnel …" % name)
        proc = None
        try:
            proc, url = TUNNELS[name]()
            if not verify_url(url):
                raise RuntimeError("URL was created but does not answer yet (blocked or still propagating)")
        except Exception as e:
            _kill(proc)
            errors.append("%s: %s" % (name, e))
            print("   ⚠️ %s failed: %s" % (name, str(e)[:200]))
            continue
        STATE.update(tproc=proc, url=url, tname=name)
        print("   ✅ %s is up" % name)
        return True
    raise FizgigError("No public tunnel could be opened",
        """
        All tunnel providers failed (listed below). Try, in this order:
        1. Re-run this cell — provider hiccups are usually temporary.
        2. Paste a free ngrok token (dashboard.ngrok.com) in cell ① → NGROK_TOKEN, re-run ① and ⑦.
        3. Make sure the runtime is connected (no “Reconnecting…” banner).""", "\n".join(errors))

def show_banner():
    url, pw = STATE["url"], PASSWORD
    js_pw = _html.escape(json.dumps(pw), quote=True)
    display(HTML(
        '<div style="font:15px system-ui;padding:16px 18px;border-radius:14px;background:#052e1a;color:#d1fae5;border:1px solid #10b981;max-width:760px">'
        '<div style="font-weight:700;font-size:17px">✅ Fizgig is live <span style="font-weight:400;opacity:.8">via %s</span></div>'
        '<div style="margin:10px 0"><a href="%s" target="_blank" style="color:#6ee7b7;font-size:18px;word-break:break-all">%s</a></div>'
        '<div>🔑 Password: <code style="background:#064e3b;padding:2px 8px;border-radius:6px;user-select:all">%s</code> '
        '<button onclick="navigator.clipboard.writeText(%s)" style="margin-left:6px;cursor:pointer">Copy</button></div>'
        '<div style="margin-top:10px;font-size:12.5px;opacity:.85">No username. Next: Preferences ▸ “Download models for me” → Start ▸ upload your images → Training ▸ Start. '
        'The link changes whenever the tunnel restarts; the newest one is always printed below.</div></div>'
        % (_html.escape(STATE["tname"]), _html.escape(url), _html.escape(url), _html.escape(pw), js_pw)))
    print("\n🌍 URL     :", url)
    print("🔑 Password:", pw)
    if STATE["tname"] == "ngrok":
        print("   (ngrok shows a one-time “Visit Site” warning page — click it.)")

# ------------------------------------------------------------------ monitor
def gpu_line():
    rc, out = run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu", "--format=csv,noheader,nounits"], quiet=True)
    try:
        u, mu, mt, tmp = [x.strip() for x in out.splitlines()[0].split(",")]
        return "%s%% %.1f/%.1f GB %s°C" % (u, float(mu) / 1024, float(mt) / 1024, tmp)
    except Exception:
        return "n/a"

def heartbeat(show_url=False):
    try:
        import psutil
        vm = psutil.virtual_memory()
        ram = "%.1f/%.1f GB" % ((vm.total - vm.available) / 2**30, vm.total / 2**30)
    except Exception:
        ram = "n/a"
    up = int(time.time() - STATE["t0"])
    print("[%s] ⏱ %dh%02dm | GPU %s | RAM %s | disk free %.0f GB | tunnel: %s | server restarts: %d"
          % (time.strftime("%H:%M:%S"), up // 3600, up % 3600 // 60, gpu_line(), ram,
             shutil.disk_usage("/content").free / 2**30, STATE["tname"], STATE["restarts"]))
    if show_url and STATE["url"]:
        print("            🌍 %s   🔑 %s" % (STATE["url"], PASSWORD))

def tunnel_alive():
    return STATE["tproc"] is not None and STATE["tproc"].poll() is None and bool(STATE["url"])

def monitor():
    print("\n🟢 Monitoring (heartbeat every 5 min). Press ■ to stop watching — Fizgig & the tunnel keep running in the background.")
    print("   Re-run this cell any time to re-attach.\n")
    last_hb = last_chk = 0
    bad_checks = 0
    try:
        while True:
            time.sleep(5)
            now = time.time()
            # --- AUTO_RESTART_SERVER
            if not server_up():
                if not CFG["AUTO_RESTART_SERVER"]:
                    print("🔴 Fizgig stopped and AUTO_RESTART_SERVER is off. Re-run this cell to start it again.")
                    print(log_tail(15))
                    return
                if STATE["restarts"] >= 30:
                    raise FizgigError("Fizgig keeps crashing (30 restarts)", "Read the log below; common causes are out-of-memory (use a lighter model / batch size 1) or a broken install (FORCE_REINSTALL).", log_tail())
                STATE["restarts"] += 1
                print("⚠️ [%s] Fizgig stopped — restarting (#%d). Last log lines:" % (time.strftime("%H:%M:%S"), STATE["restarts"]))
                print("   " + log_tail(8).replace("\n", "\n   "))
                time.sleep(min(3 * STATE["restarts"], 30))
                try:
                    start_server(); wait_ready(180)
                    print("✅ Fizgig is back. A paused/interrupted run can be continued from “Runs & outputs ▸ Continue / Resume”.")
                except FizgigError as e:
                    print("❌ restart failed:", e.problem)
                    continue
            # --- tunnel supervision
            if not tunnel_alive():
                print("⚠️ [%s] Tunnel dropped — reconnecting …" % time.strftime("%H:%M:%S"))
                try:
                    start_tunnel(); show_banner(); bad_checks = 0
                except FizgigError as e:
                    print("❌ could not reconnect:", e.problem, "— retrying in 30 s")
                    time.sleep(30)
                    continue
            elif now - last_chk > 120:
                last_chk = now
                s = http_status(STATE["url"], 10)
                bad_checks = bad_checks + 1 if (s is None or s >= 500) else 0
                if bad_checks >= 3:
                    print("⚠️ Tunnel stopped answering — reopening …")
                    try:
                        start_tunnel(); show_banner()
                    except FizgigError as e:
                        print("❌", e.problem)
                    bad_checks = 0
            # --- heartbeat
            if now - last_hb > 300:
                heartbeat(show_url=(int(now - STATE["t0"]) // 300) % 6 == 0)
                last_hb = now
    except KeyboardInterrupt:
        print("\n⏹ Stopped watching. Fizgig and the tunnel are STILL RUNNING in the background.")
        print("   Re-run this cell to resume monitoring, or use the 🧰 Utilities cell.")

# ------------------------------------------------------------------ main
if server_up():
    print("✅ Fizgig server is already running")
else:
    print("🚀 Starting Fizgig (first start can take 1–2 min) …")
    start_server()
    wait_ready()
    print("✅ Fizgig server is up on 127.0.0.1:%d" % PORT)

if tunnel_alive():
    print("🌐 Re-using the running %s tunnel" % STATE["tname"])
else:
    start_tunnel()
show_banner()
if not globals().get("KA_DONE"):
    start_keepalive()
heartbeat()
monitor()

In [ ]:
#@title 🧰 Utilities (stop cell ⑦ with ■ first — the server keeps running)  { display-mode: "form" }
ACTION = "status" #@param ["status","server log (last 80 lines)","restart server","restart tunnel","stop server + tunnel","disk usage"]

if "STATE" not in globals() or "kill_server" not in globals():
    raise FizgigError("Nothing is launched yet", "Run cell ⑦ first.")

if ACTION == "status":
    print("server up :", server_up())
    print("tunnel    :", STATE["tname"], STATE["url"])
    print("password  :", PASSWORD)
    heartbeat()
elif ACTION.startswith("server log"):
    print(log_tail(80))
elif ACTION == "restart server":
    print("⚠️ A running training stops (Fizgig ▸ Runs & outputs lets you Continue it).")
    kill_server(); time.sleep(2); start_server(); wait_ready(240); print("✅ server restarted")
elif ACTION == "restart tunnel":
    start_tunnel(); show_banner()
elif ACTION.startswith("stop"):
    stop_tunnel(); kill_server(); print("⏹ Stopped Fizgig and the tunnel. Re-run ⑦ to start again.")
elif ACTION == "disk usage":
    run("df -h /content | tail -1; du -sh /content/fizgig_models /content/fizgig_venv /content/hf_cache /content/.uv_cache /content/Fizgig 2>/dev/null")
    print("\nFree space tip: delete a model you no longer need from /content/fizgig_models, or `rm -rf /content/hf_cache`.")

In [ ]:
#@title 📦 Export trained LoRAs (run before the session ends!)  { display-mode: "form" }
EXPORT = "download as zip" #@param ["download as zip","copy to Google Drive","list files only"]

if "ensure" not in globals():
    raise RuntimeError("▶ Run the cells above first.")
ensure("APP", "MODELS", "WORKSPACE")
import zipfile

def find_loras():
    skip = {".git", "models", "cache", "venv", "__pycache__"}
    roots = [Path(os.path.realpath(APP / "output_loras")), APP / "user_data", APP]
    seen, found = set(), []
    for root in roots:
        if not root.exists():
            continue
        for dp, dn, fn in os.walk(root):
            dn[:] = [d for d in dn if d not in skip]
            for f in fn:
                if f.endswith(".safetensors"):
                    p = Path(dp) / f
                    rp = os.path.realpath(p)
                    if rp not in seen and not rp.startswith(os.path.realpath(MODELS)) and not rp.startswith(str(HF_HOME)):
                        seen.add(rp); found.append(p)
    return sorted(found, key=lambda p: p.stat().st_mtime, reverse=True)

files = find_loras()
if not files:
    print("No .safetensors outputs found yet. Train something first (Fizgig ▸ Training ▸ Start).")
else:
    total = sum(p.stat().st_size for p in files) / 2**20
    print("Found %d file(s), %.0f MB:" % (len(files), total))
    for p in files[:30]:
        print("  •", p, "(%.0f MB)" % (p.stat().st_size / 2**20))
    if EXPORT == "download as zip":
        if total > 2000:
            print("⚠️ More than 2 GB: browser downloads may fail — choose “copy to Google Drive” instead.")
        zp = "/content/fizgig_loras.zip"
        with zipfile.ZipFile(zp, "w", zipfile.ZIP_STORED) as z:
            for p in files:
                z.write(p, arcname=p.name)
        from google.colab import files as colab_files
        colab_files.download(zp)
    elif EXPORT == "copy to Google Drive":
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
        dest = Path("/content/drive/MyDrive") / CFG["DRIVE_FOLDER"] / "exported_loras" / time.strftime("%Y%m%d_%H%M%S")
        dest.mkdir(parents=True, exist_ok=True)
        for p in files:
            shutil.copy2(p, dest / p.name)
        print("✅ Copied to", dest)

## 🩺 Troubleshooting (all errors in this notebook print a “How to fix” box)

| Symptom | Fix |
|---|---|
| *No NVIDIA GPU found* | `Runtime ▸ Change runtime type ▸ T4 GPU`, then re-run from ①. |
| *Only X GB disk free* | `Runtime ▸ Disconnect and delete runtime`, start a fresh VM (~70 GB). |
| Cell ④ install fails | Re-run ④. Still failing → tick `FORCE_REINSTALL` in ①, re-run ①→④. |
| Link opens but login fails | Use the password printed in ⑦ (no username). Saved in `/content/.fizgig_password` for the session. |
| Cloudflare link not working | ⑦ verifies every URL and auto-falls back (ngrok if token → localhost.run → Pinggy). Force one with `TUNNEL` in ①. |
| *Killed* / out-of-memory | 12 GB RAM limit: pick a lighter model, batch size 1, fewer sample previews, keep `SWAP_GB` on. |
| Gated model download fails (Klein, Flux 3) | Accept the licence on Hugging Face, create a read token, set `HF_TOKEN` in ① or Colab Secrets. |
| Session ended | Everything in `/content` is gone. Use Drive (`USE_DRIVE`) for datasets/LoRAs, and run 📦 Export before leaving. |

**Notes**
* Pinggy's free tunnel lasts ~60 min per connection; ⑦ reconnects automatically (new link is printed).
* Colab free still has hard limits (≈12 h per session, GPU quota). Keep-alive only avoids *idle* disconnects.
* Update = just start a new session: ④ pulls the newest Fizgig and installs any new requirements.